# ConnectionLens — Connection Readiness Pipeline

**WID3006 ML Group Assignment: "Tying the (Data) Knot"**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iztzx/WID3006_ML/blob/main/ConnectionLens_Colab.ipynb)

## Problem Statement

Modern relationships are increasingly shaped by digital interactions — swipe patterns, message frequency, emoji usage, and online presence all leave behavioral traces. This project tackles a **multi-class classification** problem: **can we predict a user's connection-readiness stage from their dating-app behavioral signals and demographic profile?**

We define five connection-readiness stages using a composite `connection_score` (weighted blend of match quality, conversation quality, profile quality, and activity level, minus swipe excess). Users are ranked by percentile and assigned to stages based on funnel-informed thresholds. The ML pipeline trains, tunes, and compares six classifiers, applies SHAP for interpretability, and calibrates the winning model for reliable probability estimates.

**Target:** `connection_stage` — five classes:

| Stage | Selection Criteria |
|---|---|
| **Likely To Connect** | `connection_score` rank >= 0.80 |
| **Ready To Chat** | Default middle tier |
| **Mostly Browsing** | Middle tier + `browser_issue` >= 62nd pctl + `browser_issue` >= `swipe_issue` |
| **Swipes Too Freely** | Middle tier + `swipe_issue` >= 50th pctl |
| **Needs Profile Help** | `connection_score` rank <= 0.20 |

**Dataset:** [Dating App Behavior Dataset](https://www.kaggle.com/datasets/keyushnisar/dating-app-behavior-dataset) — 50,000 synthetic records with 25 features spanning demographics, app usage, and match outcomes.

## Pipeline Overview

1. Install dependencies
2. **Configuration** — centralized `CONFIG` dict with all tuneable parameters
3. Upload dataset
4. Exploratory Data Analysis (EDA)
5. Connection scoring and feature engineering (reads weights/caps from CONFIG)
6. Preprocessing and target construction
7. Train 6 models with 5-fold cross-validation
8. Hyperparameter tuning on top 3
9. Select best model, calibrate, and validate
10. SHAP interpretability
11. Classification report
12. AutoML baseline comparison (auto-sklearn / FLAML)
13. Final comparison table with all models
14. Save artifacts
15. Download artifacts (optional)
16. Launch Streamlit dashboard (optional)

**Runtime:** ~10-15 minutes on T4 GPU

## 1. Install Dependencies

In [1]:
!pip install -q pandas numpy scikit-learn imbalanced-learn xgboost lightgbm catboost shap matplotlib seaborn scipy joblib kagglehub

## 2. Configuration

All tuneable parameters — random seed, scoring weights/caps, pipeline settings, model hyperparameters, tuning search spaces, SHAP settings, and AutoML budget — are defined in a single `CONFIG` dict. Every downstream cell reads from this dict instead of using hard-coded values.

In [2]:
# ============================================================
# Centralized Configuration — single source of truth
# ============================================================

from scipy.stats import randint, uniform

CONFIG = {
    # --- Global seed (used everywhere for reproducibility) ---
    "SEED": 42,
    # --- Connection scoring ---
    "scoring": {
        "profile_completeness": {
            "caps": {"profile_pics": 6, "bio_length": 300, "interests": 5},
            "weights": {"profile_pics": 0.40, "bio_length": 0.40, "interests": 0.20},
        },
        "selectivity_balance": {"ideal_swipe_ratio": 0.55},
        "swipe_excess": {"threshold": 0.70},
        "match_quality": {
            "caps": {"matches": 40, "social_pull": 50},
            "weights": {
                "match_rate": 0.45,
                "matches": 0.25,
                "selectivity": 0.15,
                "social_pull": 0.15,
            },
        },
        "conversation_quality": {
            "caps": {"msg_per_match": 10, "messages": 80, "emoji_rate": 1, "app_usage": 300},
            "weights": {
                "msg_per_match": 0.40,
                "messages": 0.30,
                "emoji_rate": 0.20,
                "app_usage": 0.10,
            },
        },
        "profile_quality": {
            "caps": {"bio_length": 300, "profile_pics": 6},
            "weights": {"completeness": 0.60, "bio_length": 0.25, "profile_pics": 0.15},
        },
        "connection_score": {
            "weights": {
                "match_quality": 0.35,
                "conversation_quality": 0.30,
                "profile_quality": 0.20,
                "activity_level": 0.15,
                "swipe_penalty": 0.10,
            },
            "caps": {"app_usage": 300, "swipe_excess": 0.30},
        },
        "browser_issue": {
            "caps": {"app_usage": 300, "messages": 80, "matches": 40},
            "weights": {"low_app_usage": 0.45, "low_messages": 0.35, "low_matches": 0.20},
        },
        "swipe_issue": {
            "caps": {"swipe_excess": 0.30},
            "weights": {"swipe_excess": 0.55, "low_match_rate": 0.45},
        },
        "stage_thresholds": {
            "likely_to_connect": 0.80,
            "needs_profile_help": 0.20,
            "mostly_browsing_quantile": 0.62,
            "swipes_too_freely_quantile": 0.50,
        },
    },
    # --- ML pipeline ---
    "pipeline": {
        "test_size": 0.2,
        "cv_folds": 5,
        "tuning_cv_folds": 3,
        "calibration_cv_folds": 3,
        "feature_selection_cumulative_threshold": 0.95,
        "feature_selection_min_features": 20,
        "selector_rf_n_estimators": 300,
        "tuning_n_iter": 20,
    },
    # --- Base model hyperparameters ---
    "models": {
        "Logistic Regression": {"max_iter": 2000},
        "Random Forest": {"n_estimators": 300, "max_depth": 20},
        "Gradient Boosting": {"n_estimators": 100, "max_depth": 5, "learning_rate": 0.1},
        "XGBoost": {
            "n_estimators": 300,
            "max_depth": 6,
            "learning_rate": 0.1,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.1,
        },
        "LightGBM": {
            "n_estimators": 300,
            "max_depth": 8,
            "learning_rate": 0.1,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.1,
        },
        "CatBoost": {"iterations": 300, "depth": 6, "learning_rate": 0.1},
    },
    # --- Tuning search spaces (scipy.stats distribution args) ---
    "tuning": {
        "Logistic Regression": {"C": (0.01, 99.99)},
        "Random Forest": {
            "n_estimators": (200, 600),
            "max_depth": (10, 40),
            "min_samples_split": (2, 15),
            "min_samples_leaf": (1, 10),
        },
        "Gradient Boosting": {
            "n_estimators": (100, 400),
            "max_depth": (3, 10),
            "learning_rate": (0.01, 0.29),
            "subsample": (0.6, 0.4),
        },
        "XGBoost": {
            "n_estimators": (200, 600),
            "max_depth": (3, 10),
            "learning_rate": (0.01, 0.29),
            "subsample": (0.6, 0.4),
            "colsample_bytree": (0.6, 0.4),
            "reg_alpha": (0.001, 9.999),
        },
        "LightGBM": {
            "n_estimators": (200, 600),
            "max_depth": (4, 12),
            "learning_rate": (0.01, 0.29),
            "subsample": (0.6, 0.4),
            "colsample_bytree": (0.6, 0.4),
        },
        "CatBoost": {
            "iterations": (200, 600),
            "depth": (4, 10),
            "learning_rate": (0.01, 0.29),
            "l2_leaf_reg": (0.001, 9.999),
        },
    },
    # --- SHAP ---
    "shap": {
        "sample_size": 1000,
        "model_n_estimators": 200,
        "model_max_depth": 15,
        "max_display": 20,
    },
    # --- AutoML ---
    "automl": {
        "time_budget": 300,
    },
}

SEED = CONFIG["SEED"]
print("CONFIG loaded.")
print(f"  Seed: {SEED}")
print(f"  Models: {list(CONFIG['models'].keys())}")
print(f"  CV folds: {CONFIG['pipeline']['cv_folds']}")

CONFIG loaded.
  Seed: 42
  Models: ['Logistic Regression', 'Random Forest', 'Gradient Boosting', 'XGBoost', 'LightGBM', 'CatBoost']
  CV folds: 5


## 3. Load Dataset

Downloads `dating_app_behavior_dataset_extended1.csv` from Kaggle via `kagglehub`. 
Falls back to a local file if already present.

**Colab:** Add your Kaggle credentials via Colab Secrets (🔑 icon) or `~/.kaggle/kaggle.json`.

**Local:** Place `kaggle.json` in `~/.kaggle/` or set `KAGGLE_USERNAME` / `KAGGLE_KEY` env vars.

In [3]:
import shutil
from pathlib import Path

import kagglehub
from kagglehub import KaggleDatasetAdapter

DATASET_PATH = Path("dating_app_behavior_dataset_extended1.csv")
KAGGLE_DATASET = "keyushnisar/dating-app-behavior-dataset"
KAGGLE_FILE = "dating_app_behavior_dataset_extended1.csv"

_dataset_loaded = False

# --- Method 1: Load directly into pandas ---
try:
    _df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS, KAGGLE_DATASET, KAGGLE_FILE
    )
    _df.to_csv(DATASET_PATH, index=False)
    _dataset_loaded = True
    print(f"Dataset loaded via kagglehub.dataset_load: {_df.shape}")
except Exception as e:
    print(f"kagglehub.dataset_load failed: {e}")

# --- Method 2: Download to cache, copy file ---
if not _dataset_loaded:
    try:
        download_dir = kagglehub.dataset_download(KAGGLE_DATASET)
        src = Path(download_dir) / KAGGLE_FILE
        if src.exists():
            shutil.copy2(src, DATASET_PATH)
            _dataset_loaded = True
            print(f"Dataset downloaded via kagglehub.dataset_download: {DATASET_PATH}")
        else:
            print(f"Downloaded to {download_dir}, contents: {list(Path(download_dir).iterdir())}")
    except Exception as e:
        print(f"kagglehub.dataset_download failed: {e}")

# --- Method 3: Use existing local file ---
if not _dataset_loaded and DATASET_PATH.exists():
    print(f"Using existing local file: {DATASET_PATH}")
    _dataset_loaded = True

if not _dataset_loaded:
    raise FileNotFoundError(
        "Could not obtain dating_app_behavior_dataset_extended1.csv. "
        "Configure Kaggle auth: ~/.kaggle/kaggle.json or "
        "KAGGLE_USERNAME + KAGGLE_KEY env vars."
    )

print(f"Dataset ready: {DATASET_PATH}")

c:\WID3006_ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset loaded via kagglehub.dataset_load: (50000, 25)
Dataset ready: dating_app_behavior_dataset_extended1.csv


## 4. Exploratory Data Analysis (EDA)

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

# Load raw data for EDA
eda_df = pd.read_csv(DATASET_PATH)

print(f"Dataset shape: {eda_df.shape}")
print(f"\nColumn types:\n{eda_df.dtypes}")
print(f"\nMissing values per column:\n{eda_df.isnull().sum()}")
print(f"\nBasic statistics:\n{eda_df.describe().round(2)}")

In [5]:
# --- 1. Gender and Income distributions ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if "gender" in eda_df.columns:
    sns.countplot(x="gender", data=eda_df, palette="Set3", ax=axes[0])
    axes[0].set_title("Gender Distribution")

if "income_bracket" in eda_df.columns:
    income_order = [
        "Very Low",
        "Low",
        "Lower-Middle",
        "Middle",
        "Upper-Middle",
        "High",
        "Very High",
    ]
    sns.countplot(
        x="income_bracket",
        data=eda_df,
        order=income_order,
        palette="coolwarm",
        ax=axes[1],
    )
    axes[1].set_title("Income Bracket Distribution")
    axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# --- 2. Behavioral feature distributions ---
behav_cols = [
    "app_usage_time_min",
    "swipe_right_ratio",
    "message_sent_count",
    "likes_received",
    "emoji_usage_rate",
    "mutual_matches",
]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, col in enumerate(behav_cols):
    row, col_idx = divmod(idx, 3)
    if col in eda_df.columns:
        sns.histplot(
            eda_df[col], bins=40, kde=True, ax=axes[row][col_idx], color="steelblue"
        )
        axes[row][col_idx].set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

# --- 3. Correlation heatmap of numeric features ---
numeric_cols = eda_df.select_dtypes(include=[np.number]).columns.tolist()
corr = eda_df[numeric_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(
    corr,
    cmap="RdBu_r",
    center=0,
    annot=False,
    fmt=".2f",
    square=True,
    linewidths=0.5,
)
plt.title("Correlation Heatmap of Numeric Features", fontsize=14)
plt.tight_layout()
plt.show()

# --- 4. Interest tags analysis ---
if "interest_tags" in eda_df.columns:
    tags = eda_df["interest_tags"].fillna("").str.split(r",\s*")
    tag_counts = tags.explode().value_counts()
    tag_counts = tag_counts[tag_counts.index != ""].head(15)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=tag_counts.values, y=tag_counts.index, palette="viridis")
    plt.title("Top 15 Interest Tags", fontsize=14)
    plt.xlabel("Count")
    plt.tight_layout()
    plt.show()

print(f"\nEDA complete — {len(eda_df)} records, {eda_df.shape[1]} columns")
print(f"Missing values: {eda_df.isnull().sum().sum()}")

C:\Users\tzx20\AppData\Local\Temp\ipykernel_31068\4238497606.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\tzx20\AppData\Local\Temp\ipykernel_31068\4238497606.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\tzx20\AppData\Local\Temp\ipykernel_31068\4238497606.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



EDA complete — 50000 records, 25 columns
Missing values: 0


C:\Users\tzx20\AppData\Local\Temp\ipykernel_31068\4238497606.py:78: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Connection Scoring and Feature Engineering

All scoring logic is inlined below — no external modules required. 
Weights, caps, and thresholds are read from `CONFIG["scoring"]`.

In [6]:
# ===========================================================================
# Connection Scoring — fully inlined (no external module imports)
# ===========================================================================

TARGET_COL = "connection_stage"

STAGE_LABELS = [
    "Needs Profile Help",
    "Mostly Browsing",
    "Swipes Too Freely",
    "Ready To Chat",
    "Likely To Connect",
]

STRONG_STAGES = {"Ready To Chat", "Likely To Connect"}


# -- Helpers -----------------------------------------------------------------

def _numeric(frame, column, default=0.0):
    if column not in frame.columns:
        return pd.Series(default, index=frame.index, dtype="float64")
    return pd.to_numeric(frame[column], errors="coerce").fillna(default)


def _rank01(series):
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    if values.nunique(dropna=False) <= 1:
        return pd.Series(0.5, index=values.index, dtype="float64")
    return values.rank(pct=True).clip(0, 1)


def _bounded(series, cap):
    values = pd.to_numeric(series, errors="coerce").fillna(0).clip(lower=0)
    return (values / cap).clip(0, 1)


def _num_interests(frame):
    if "interest_tags" in frame.columns:
        tags = frame["interest_tags"].fillna("").astype(str).str.split(r",\s*")
        return tags.apply(lambda values: len([tag for tag in values if tag]))
    if "num_interests" in frame.columns:
        return _numeric(frame, "num_interests")
    return pd.Series(0, index=frame.index, dtype="float64")


# -- Feature Engineering -----------------------------------------------------

def add_connection_features(frame, config=None):
    """Add interpretable product features. Reads weights/caps from CONFIG["scoring"]."""
    cfg = config if config is not None else CONFIG["scoring"]
    engineered = frame.copy()

    app_usage = _numeric(engineered, "app_usage_time_min")
    swipe_ratio = _numeric(engineered, "swipe_right_ratio")
    likes = _numeric(engineered, "likes_received")
    matches = _numeric(engineered, "mutual_matches")
    messages = _numeric(engineered, "message_sent_count")
    emoji_rate = _numeric(engineered, "emoji_usage_rate")
    bio_length = _numeric(engineered, "bio_length")
    profile_pics = _numeric(engineered, "profile_pics_count")
    height = _numeric(engineered, "height_cm")
    weight = _numeric(engineered, "weight_kg")

    # Profile completeness
    pc_caps = cfg["profile_completeness"]["caps"]
    pc_wts = cfg["profile_completeness"]["weights"]
    engineered["num_interests"] = _num_interests(engineered)
    engineered["match_rate"] = matches / (likes + 1)
    engineered["msg_per_match"] = messages / (matches + 1)
    if "height_cm" in frame.columns and "weight_kg" in frame.columns:
        engineered["bmi"] = np.where(height > 0, weight / ((height / 100) ** 2), 0)
    engineered["profile_completeness"] = (
        profile_pics.clip(0, pc_caps["profile_pics"]) / pc_caps["profile_pics"] * pc_wts["profile_pics"]
        + bio_length.clip(0, pc_caps["bio_length"]) / pc_caps["bio_length"] * pc_wts["bio_length"]
        + engineered["num_interests"].clip(0, pc_caps["interests"]) / pc_caps["interests"] * pc_wts["interests"]
    )

    # Selectivity and swipe behavior
    ideal_ratio = cfg["selectivity_balance"]["ideal_swipe_ratio"]
    swipe_thresh = cfg["swipe_excess"]["threshold"]
    engineered["selectivity_balance"] = (1 - (swipe_ratio - ideal_ratio).abs() / ideal_ratio).clip(0, 1)
    engineered["swipe_excess"] = (swipe_ratio - swipe_thresh).clip(lower=0)
    engineered["like_to_match_gap"] = (likes - matches).clip(lower=0)
    engineered["conversation_depth"] = np.log1p(messages) * np.log1p(engineered["msg_per_match"])
    engineered["social_pull"] = likes / (profile_pics + 1)
    engineered["activity_level"] = np.log1p(app_usage)

    if "last_active_hour" in engineered.columns:
        hour = _numeric(engineered, "last_active_hour") % 24
        engineered["last_active_sin"] = np.sin(2 * np.pi * hour / 24)
        engineered["last_active_cos"] = np.cos(2 * np.pi * hour / 24)

    # Match quality
    mq_caps = cfg["match_quality"]["caps"]
    mq_wts = cfg["match_quality"]["weights"]
    engineered["match_quality"] = (
        mq_wts["match_rate"] * engineered["match_rate"].clip(0, 1)
        + mq_wts["matches"] * _bounded(matches, mq_caps["matches"])
        + mq_wts["selectivity"] * engineered["selectivity_balance"]
        + mq_wts["social_pull"] * _bounded(engineered["social_pull"], mq_caps["social_pull"])
    )

    # Conversation quality
    cq_caps = cfg["conversation_quality"]["caps"]
    cq_wts = cfg["conversation_quality"]["weights"]
    engineered["conversation_quality"] = (
        cq_wts["msg_per_match"] * _bounded(engineered["msg_per_match"], cq_caps["msg_per_match"])
        + cq_wts["messages"] * _bounded(messages, cq_caps["messages"])
        + cq_wts["emoji_rate"] * _bounded(emoji_rate, cq_caps["emoji_rate"])
        + cq_wts["app_usage"] * _bounded(app_usage, cq_caps["app_usage"])
    )

    # Profile quality
    pq_caps = cfg["profile_quality"]["caps"]
    pq_wts = cfg["profile_quality"]["weights"]
    engineered["profile_quality"] = (
        pq_wts["completeness"] * engineered["profile_completeness"]
        + pq_wts["bio_length"] * _bounded(bio_length, pq_caps["bio_length"])
        + pq_wts["profile_pics"] * _bounded(profile_pics, pq_caps["profile_pics"])
    )

    # Connection score
    cs_caps = cfg["connection_score"]["caps"]
    cs_wts = cfg["connection_score"]["weights"]
    engineered["connection_score"] = (
        cs_wts["match_quality"] * engineered["match_quality"]
        + cs_wts["conversation_quality"] * engineered["conversation_quality"]
        + cs_wts["profile_quality"] * engineered["profile_quality"]
        + cs_wts["activity_level"] * _bounded(app_usage, cs_caps["app_usage"])
        - cs_wts["swipe_penalty"] * _bounded(engineered["swipe_excess"], cs_caps["swipe_excess"])
    )

    # Browser / swipe issue signals
    bi_caps = cfg["browser_issue"]["caps"]
    bi_wts = cfg["browser_issue"]["weights"]
    engineered["browser_issue"] = (
        bi_wts["low_app_usage"] * (1 - _bounded(app_usage, bi_caps["app_usage"]))
        + bi_wts["low_messages"] * (1 - _bounded(messages, bi_caps["messages"]))
        + bi_wts["low_matches"] * (1 - _bounded(matches, bi_caps["matches"]))
    )

    si_caps = cfg["swipe_issue"]["caps"]
    si_wts = cfg["swipe_issue"]["weights"]
    engineered["swipe_issue"] = (
        si_wts["swipe_excess"] * _bounded(engineered["swipe_excess"], si_caps["swipe_excess"])
        + si_wts["low_match_rate"] * (1 - engineered["match_rate"].clip(0, 1))
    )

    return engineered


# -- Target Construction -----------------------------------------------------

def construct_connection_stage(frame, config=None):
    """Create five plain-language connection-readiness labels from funnel signals."""
    cfg = config if config is not None else CONFIG["scoring"]
    thresholds = cfg["stage_thresholds"]
    scored = add_connection_features(frame, config=cfg)

    score_rank = _rank01(scored["connection_score"])
    browser_issue = scored["browser_issue"]
    swipe_issue = scored["swipe_issue"]

    labels = pd.Series("Ready To Chat", index=scored.index, dtype="object")
    labels[score_rank >= thresholds["likely_to_connect"]] = "Likely To Connect"
    labels[score_rank <= thresholds["needs_profile_help"]] = "Needs Profile Help"

    middle = labels.eq("Ready To Chat")
    labels[
        middle
        & (browser_issue >= browser_issue.quantile(thresholds["mostly_browsing_quantile"]))
        & (browser_issue >= swipe_issue)
    ] = "Mostly Browsing"

    middle = labels.eq("Ready To Chat")
    labels[
        middle & (swipe_issue >= swipe_issue.quantile(thresholds["swipes_too_freely_quantile"]))
    ] = "Swipes Too Freely"

    ordered = pd.Categorical(labels, categories=STAGE_LABELS, ordered=True)
    return pd.Series(ordered, index=labels.index).astype(str)


print("Connection scoring inlined (no external modules).")
print(f"Target: {TARGET_COL}")
print(f"Stages: {STAGE_LABELS}")

Connection scoring inlined (no external modules).
Target: connection_stage
Stages: ['Needs Profile Help', 'Mostly Browsing', 'Swipes Too Freely', 'Ready To Chat', 'Likely To Connect']


## 6. Preprocessing and Target Construction

In [7]:
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

PIPE = CONFIG["pipeline"]

# --- Load ---
df = pd.read_csv(DATASET_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Missing values: {df.isnull().sum().sum()}")

# --- Construct 5-class target ---
print(f"\nConstructing {TARGET_COL} target...")
df[TARGET_COL] = construct_connection_stage(df)
dist = df[TARGET_COL].value_counts().sort_index()
print(f"Connection-stage distribution:\n{dist.to_dict()}")

# --- Feature engineering ---
print("\nEngineering features...")
df = add_connection_features(df)

# Interest tags -> binary columns
tags_series = df["interest_tags"].fillna("").str.split(r",\s*")
all_tags = set()
for tag_list in tags_series:
    all_tags.update(tag_list)
all_tags.discard("")
sorted_tags = sorted(all_tags)
print(f"  Found {len(sorted_tags)} unique interest tags")

for tag in sorted_tags:
    df[f"tag_{tag}"] = tags_series.apply(lambda x: 1 if tag in x else 0)

# Drop redundant columns
cols_to_drop = [
    "interest_tags",
    "app_usage_time_label",
    "swipe_right_label",
    "relationship_intent",
    "match_outcome",
    "connection_score",
    "browser_issue",
    "swipe_issue",
    "engagement_score",
]
dropped = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=dropped)
print(f"  Dropped: {dropped}")

# --- Encode categoricals ---
print("\nEncoding categorical features...")
y = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL])

income_order = [
    "Very Low",
    "Low",
    "Lower-Middle",
    "Middle",
    "Upper-Middle",
    "High",
    "Very High",
]
education_order = [
    "No Formal Education",
    "High School",
    "Associate's",
    "Bachelor's",
    "Master's",
    "Postdoc",
    "PhD",
]

for col, order in [
    ("income_bracket", income_order),
    ("education_level", education_order),
]:
    if col in X.columns:
        mapping = {v: i for i, v in enumerate(order)}
        X[col] = X[col].map(mapping).fillna(-1).astype(int)
print("  Ordinal encoded: income_bracket, education_level")

remaining_cat = X.select_dtypes(include=["object"]).columns.tolist()
if remaining_cat:
    X = pd.get_dummies(X, columns=remaining_cat, drop_first=True)
    print(f"  One-hot encoded: {remaining_cat}")

print(f"  Final feature count: {X.shape[1]}")

# --- Encode target ---
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)
print(f"  Target classes: {list(target_encoder.classes_)}")

# --- Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=PIPE["test_size"], random_state=SEED, stratify=y_encoded
)
print(f"  Train: {X_train.shape} | Test: {X_test.shape}")

# --- Scale ---
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)

# --- Feature selection via RF importance ---
print("  Selecting features via RF importance...")
selector_rf = RandomForestClassifier(
    n_estimators=PIPE["selector_rf_n_estimators"], random_state=SEED, n_jobs=-1
)
selector_rf.fit(X_train_scaled, y_train)

importances = selector_rf.feature_importances_
importance_df = pd.DataFrame(
    {"Feature": X_train_scaled.columns, "Importance": importances}
).sort_values("Importance", ascending=False)
importance_df["Cumulative"] = importance_df["Importance"].cumsum()
threshold = PIPE["feature_selection_cumulative_threshold"] * importance_df["Importance"].sum()
selected_features = importance_df[importance_df["Cumulative"] <= threshold][
    "Feature"
].tolist()
if len(selected_features) < PIPE["feature_selection_min_features"]:
    selected_features = importance_df.head(PIPE["feature_selection_min_features"])["Feature"].tolist()

X_train_final = X_train_scaled[selected_features]
X_test_final = X_test_scaled[selected_features]
print(f"  Selected {len(selected_features)} / {X_train_scaled.shape[1]} features")
print(f"  Train: {X_train_final.shape} | Test: {X_test_final.shape}")

# --- Save artifacts ---
OUT_DIR = Path("Preprocessed_Data_V2")
OUT_DIR.mkdir(exist_ok=True)
X_train_final.to_csv(OUT_DIR / "X_train_selected_unresampled.csv", index=False)
X_test_final.to_csv(OUT_DIR / "X_test_selected.csv", index=False)
pd.DataFrame(y_train, columns=[TARGET_COL]).to_csv(
    OUT_DIR / "y_train_original.csv", index=False
)
pd.DataFrame(y_test, columns=[TARGET_COL]).to_csv(OUT_DIR / "y_test.csv", index=False)
joblib.dump(scaler, OUT_DIR / "scaler.pkl")
joblib.dump(target_encoder, OUT_DIR / "target_encoder.pkl")
joblib.dump(selected_features, OUT_DIR / "selected_features.pkl")

# Plot top features
plt.figure(figsize=(10, 8))
sns.barplot(
    x="Importance",
    y="Feature",
    data=importance_df.head(20),
    palette="magma",
    hue="Feature",
    legend=False,
)
plt.title("Top 20 Feature Importances (Unbiased)", fontsize=14)
plt.tight_layout()
plt.show()

print("\nPreprocessing complete!")

Dataset shape: (50000, 25)
Columns: ['gender', 'sexual_orientation', 'location_type', 'income_bracket', 'education_level', 'interest_tags', 'app_usage_time_min', 'app_usage_time_label', 'swipe_right_ratio', 'swipe_right_label', 'likes_received', 'mutual_matches', 'profile_pics_count', 'bio_length', 'message_sent_count', 'emoji_usage_rate', 'last_active_hour', 'swipe_time_of_day', 'match_outcome', 'age', 'height_cm', 'weight_kg', 'zodiac_sign', 'body_type', 'relationship_intent']
Missing values: 0

Constructing connection_stage target...
Connection-stage distribution:
{'Likely To Connect': 10001, 'Mostly Browsing': 9761, 'Needs Profile Help': 10000, 'Ready To Chat': 10419, 'Swipes Too Freely': 9819}

Engineering features...
  Found 49 unique interest tags
  Dropped: ['interest_tags', 'app_usage_time_label', 'swipe_right_label', 'relationship_intent', 'match_outcome', 'connection_score', 'browser_issue', 'swipe_issue']

Encoding categorical features...
  Ordinal encoded: income_bracket, 

C:\Users\tzx20\AppData\Local\Temp\ipykernel_31068\2833257091.py:85: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  remaining_cat = X.select_dtypes(include=["object"]).columns.tolist()


  Selecting features via RF importance...
  Selected 66 / 117 features
  Train: (40000, 66) | Test: (10000, 66)

Preprocessing complete!


C:\Users\tzx20\AppData\Local\Temp\ipykernel_31068\2833257091.py:161: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Train 6 Models with 5-Fold CV

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

PIPE = CONFIG["pipeline"]
MODEL_CFG = CONFIG["models"]

cv = StratifiedKFold(n_splits=PIPE["cv_folds"], shuffle=True, random_state=SEED)


def make_pipeline(model):
    """Wrap model in SMOTE-in-pipeline (leakage-free CV)."""
    return Pipeline([("smote", SMOTE(random_state=SEED)), ("model", model)])


models = {
    "Logistic Regression": make_pipeline(
        LogisticRegression(**MODEL_CFG["Logistic Regression"], random_state=SEED)
    ),
    "Random Forest": make_pipeline(
        RandomForestClassifier(**MODEL_CFG["Random Forest"], random_state=SEED, n_jobs=-1)
    ),
    "Gradient Boosting": make_pipeline(
        GradientBoostingClassifier(**MODEL_CFG["Gradient Boosting"], random_state=SEED)
    ),
    "XGBoost": make_pipeline(
        XGBClassifier(
            **MODEL_CFG["XGBoost"],
            random_state=SEED,
            n_jobs=-1,
            verbosity=0,
            eval_metric="mlogloss",
        )
    ),
    "LightGBM": make_pipeline(
        LGBMClassifier(
            **MODEL_CFG["LightGBM"],
            random_state=SEED,
            n_jobs=-1,
            verbose=-1,
        )
    ),
    "CatBoost": make_pipeline(
        CatBoostClassifier(**MODEL_CFG["CatBoost"], random_state=SEED, verbose=0)
    ),
}

base_results = []

for name, pipeline in models.items():
    print(f"Training {name}...", end=" ")
    t0 = pd.Timestamp.now()

    cv_scores = cross_val_score(
        pipeline, X_train_final, y_train, cv=cv, scoring="accuracy", n_jobs=-1
    )
    pipeline.fit(X_train_final, y_train)
    y_pred = pipeline.predict(X_test_final)
    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred, average="weighted")
    elapsed = (pd.Timestamp.now() - t0).total_seconds()

    base_results.append(
        {
            "Model": name,
            "CV Accuracy (mean)": round(cv_scores.mean(), 4),
            "CV Accuracy (std)": round(cv_scores.std(), 4),
            "Test Accuracy": round(test_acc, 4),
            "Test F1 (weighted)": round(test_f1, 4),
            "Train Time (s)": round(elapsed, 1),
        }
    )
    print(
        f"CV: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f} | "
        f"Test: {test_acc:.4f} | F1: {test_f1:.4f} | {elapsed:.1f}s"
    )

base_results_df = pd.DataFrame(base_results).sort_values(
    "Test Accuracy", ascending=False
)
print("\n--- Base Model Comparison ---")
print(base_results_df.to_string(index=False))

Training Logistic Regression... CV: 0.9464 +/- 0.0023 | Test: 0.9461 | F1: 0.9460 | 13.3s
Training Random Forest... CV: 0.9021 +/- 0.0044 | Test: 0.9051 | F1: 0.9050 | 39.5s
Training Gradient Boosting... CV: 0.9345 +/- 0.0036 | Test: 0.9391 | F1: 0.9392 | 478.7s
Training XGBoost... CV: 0.9515 +/- 0.0029 | Test: 0.9552 | F1: 0.9552 | 39.3s
Training LightGBM... CV: 0.9511 +/- 0.0033 | Test: 0.9553 | F1: 0.9553 | 36.1s
Training CatBoost... CV: 0.9529 +/- 0.0013 | Test: 0.9580 | F1: 0.9580 | 46.6s

--- Base Model Comparison ---
              Model  CV Accuracy (mean)  CV Accuracy (std)  Test Accuracy  Test F1 (weighted)  Train Time (s)
           CatBoost              0.9529             0.0013         0.9580              0.9580            46.6
           LightGBM              0.9510             0.0033         0.9553              0.9553            36.1
            XGBoost              0.9514             0.0029         0.9552              0.9552            39.3
Logistic Regression           

## 8. Hyperparameter Tuning (Top 3)

In [9]:
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV

TUNE_CFG = CONFIG["tuning"]
PIPE = CONFIG["pipeline"]

# Select top 3 by CV accuracy (not test accuracy — prevents leakage)
top3_names = (
    pd.DataFrame(base_results)
    .sort_values("CV Accuracy (mean)", ascending=False)
    .head(3)["Model"]
    .tolist()
)
print(f"Tuning top 3 (by CV): {top3_names}")


def _build_search_space(model_name):
    """Convert CONFIG tuning tuples to scipy distributions."""
    raw = TUNE_CFG[model_name]
    space = {}
    for param, args in raw.items():
        if isinstance(args, tuple):
            if isinstance(args[0], int):
                space[f"model__{param}"] = randint(*args)
            else:
                space[f"model__{param}"] = uniform(*args)
        else:
            space[f"model__{param}"] = args
    return space


# Logistic Regression needs extra fixed params
_lr_space = _build_search_space("Logistic Regression")
_lr_space["model__penalty"] = ["l2"]
_lr_space["model__solver"] = ["lbfgs"]

param_distributions = {name: _build_search_space(name) for name in TUNE_CFG}
param_distributions["Logistic Regression"] = _lr_space

tuned_results = {}
tuned_pipelines = {}

for name in top3_names:
    print(f"\nTuning {name}...", end=" ")
    t0 = pd.Timestamp.now()

    search = RandomizedSearchCV(
        models[name],
        param_distributions=param_distributions[name],
        n_iter=PIPE["tuning_n_iter"],
        cv=PIPE["tuning_cv_folds"],
        scoring="accuracy",
        random_state=SEED,
        n_jobs=-1,
        verbose=0,
    )
    search.fit(X_train_final, y_train)

    best_pipe = search.best_estimator_
    y_pred = best_pipe.predict(X_test_final)
    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred, average="weighted")
    elapsed = (pd.Timestamp.now() - t0).total_seconds()

    tuned_results[name] = {
        "best_params": search.best_params_,
        "cv_score": round(search.best_score_, 4),
        "test_acc": round(test_acc, 4),
        "test_f1": round(test_f1, 4),
    }
    tuned_pipelines[name] = best_pipe
    print(
        f"CV: {search.best_score_:.4f} | Test: {test_acc:.4f} | "
        f"F1: {test_f1:.4f} | {elapsed:.1f}s | {search.best_params_}"
    )

print("\nTuning complete!")

Tuning top 3 (by CV): ['CatBoost', 'XGBoost', 'LightGBM']

Tuning CatBoost... CV: 0.9585 | Test: 0.9653 | F1: 0.9653 | 1213.0s | {'model__depth': 6, 'model__iterations': 563, 'model__l2_leaf_reg': np.float64(5.142830149697703), 'model__learning_rate': np.float64(0.18180022496999232)}

Tuning XGBoost... CV: 0.9511 | Test: 0.9547 | F1: 0.9547 | 264.0s | {'model__colsample_bytree': np.float64(0.8347004662655393), 'model__learning_rate': np.float64(0.28992403910660003), 'model__max_depth': 6, 'model__n_estimators': 579, 'model__reg_alpha': np.float64(2.7607158210434113), 'model__subsample': np.float64(0.718509402281633)}

Tuning LightGBM... CV: 0.9494 | Test: 0.9552 | F1: 0.9552 | 382.8s | {'model__colsample_bytree': np.float64(0.9140703845572055), 'model__learning_rate': np.float64(0.06790539682592432), 'model__max_depth': 10, 'model__n_estimators': 443, 'model__subsample': np.float64(0.836965827544817)}

Tuning complete!


## 9. Select Best Model, Calibrate, and Validate

In [10]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

PIPE = CONFIG["pipeline"]

# Find best by CV accuracy (not test accuracy)
all_candidates = []
for r in base_results:
    all_candidates.append(
        {
            "name": r["Model"],
            "selection_score": r["CV Accuracy (mean)"],
            "test_acc": r["Test Accuracy"],
            "source": "base",
        }
    )
for name, r in tuned_results.items():
    all_candidates.append(
        {
            "name": name,
            "selection_score": r["cv_score"],
            "test_acc": r["test_acc"],
            "source": "tuned",
        }
    )

best_candidate = max(all_candidates, key=lambda x: x["selection_score"])
best_name = best_candidate["name"]
print(
    f"Best model: {best_name} ({best_candidate['source']}) — "
    f"CV: {best_candidate['selection_score']:.4f} | "
    f"Test Acc: {best_candidate['test_acc']:.4f}"
)

# Get pipeline
if best_name in tuned_pipelines:
    best_pipeline = tuned_pipelines[best_name]
else:
    best_pipeline = models[best_name]

# Calibrate — build a fresh estimator with native Python types to avoid clone() failure
fitted_estimator = best_pipeline.named_steps["model"]

def _build_fresh_estimator(estimator):
    """Create a new unfitted estimator with numpy params converted to native types."""
    params = estimator.get_params(deep=False)
    native = {}
    for k, v in params.items():
        if isinstance(v, (np.floating, np.integer)):
            native[k] = v.item()
        elif isinstance(v, np.bool_):
            native[k] = bool(v)
        else:
            native[k] = v
    return type(estimator)(**native)

base_estimator = _build_fresh_estimator(fitted_estimator)

calibrated_pipeline = Pipeline(
    [
        ("smote", SMOTE(random_state=SEED)),
        ("model", CalibratedClassifierCV(
            base_estimator, method="sigmoid", cv=PIPE["calibration_cv_folds"]
        )),
    ]
)
calibrated_pipeline.fit(X_train_final, y_train)

y_pred_cal = calibrated_pipeline.predict(X_test_final)
y_proba_cal = calibrated_pipeline.predict_proba(X_test_final)
cal_acc = accuracy_score(y_test, y_pred_cal)
cal_f1 = f1_score(y_test, y_pred_cal, average="weighted")
print(f"Calibrated — Accuracy: {cal_acc:.4f}, F1: {cal_f1:.4f}")

# Calibration plot — one-vs-rest for all classes
n_classes = y_proba_cal.shape[1]
class_names_cal = [str(c) for c in target_encoder.classes_]

plt.figure(figsize=(8, 8))
plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfectly calibrated")
for cls_idx in range(n_classes):
    fraction_of_positives, mean_predicted_value = calibration_curve(
        (y_test == cls_idx).astype(int),
        y_proba_cal[:, cls_idx],
        n_bins=10,
        strategy="uniform",
    )
    plt.plot(
        mean_predicted_value,
        fraction_of_positives,
        "s-",
        label=f"Class {class_names_cal[cls_idx]}",
    )
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Fraction of Positives")
plt.title(f"Calibration Plot — {best_name} (All Classes)")
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Best model: CatBoost (tuned) — CV: 0.9585 | Test Acc: 0.9653
Calibrated — Accuracy: 0.9644, F1: 0.9644


C:\Users\tzx20\AppData\Local\Temp\ipykernel_31068\2843779602.py:99: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. SHAP Interpretability

We compute SHAP values on the **best model** (CatBoost) using `shap.TreeExplainer`, which supports CatBoost natively.

In [11]:
import shap

SHAP_CFG = CONFIG["shap"]

# Use the best model directly for SHAP (TreeExplainer supports CatBoost natively)
shap_estimator = best_pipeline.named_steps["model"]

shap_sample = X_train_final.sample(
    n=min(SHAP_CFG["sample_size"], len(X_train_final)), random_state=SEED
)
explainer = shap.TreeExplainer(shap_estimator)
shap_values = explainer.shap_values(shap_sample)

# CatBoost multi-class returns shape (n_samples, n_features, n_classes)
if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    sv_beeswarm = shap_values[:, :, 0]       # first class for beeswarm
    sv_bar = np.mean(np.abs(shap_values), axis=2)  # mean |SHAP| for bar
elif isinstance(shap_values, list):
    sv_beeswarm = shap_values[0]
    sv_bar = sv_beeswarm
else:
    sv_beeswarm = shap_values
    sv_bar = sv_beeswarm

# Beeswarm plot
print("SHAP Beeswarm Plot:")
shap.summary_plot(sv_beeswarm, shap_sample, show=False, max_display=SHAP_CFG["max_display"])
plt.title(f"SHAP Feature Importance — {best_name}", fontsize=13)
plt.tight_layout()
plt.show()

# Bar plot
print("\nSHAP Bar Plot:")
shap.summary_plot(sv_bar, shap_sample, plot_type="bar", show=False, max_display=SHAP_CFG["max_display"])
plt.title(f"SHAP Mean |SHAP| — {best_name}", fontsize=13)
plt.tight_layout()
plt.show()

# Feature importance from the SHAP model
importances_shap = shap_estimator.feature_importances_
importance_shap_df = pd.DataFrame(
    {"Feature": X_train_final.columns, "Importance": importances_shap}
).sort_values("Importance", ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(
    x="Importance",
    y="Feature",
    data=importance_shap_df.head(SHAP_CFG["max_display"]),
    palette="magma",
    hue="Feature",
    legend=False,
)
plt.title("Top 20 Feature Importances (CatBoost)", fontsize=14)
plt.tight_layout()
plt.show()

SHAP Beeswarm Plot:


C:\Users\tzx20\AppData\Local\Temp\ipykernel_31068\1626303589.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\tzx20\AppData\Local\Temp\ipykernel_31068\1626303589.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



SHAP Bar Plot:


C:\Users\tzx20\AppData\Local\Temp\ipykernel_31068\1626303589.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Classification Report

In [12]:
# Build comparison table (AutoML row will be appended in the next section)
final_results = base_results.copy()

for name, r in tuned_results.items():
    final_results.append(
        {
            "Model": f"{name} (tuned)",
            "CV Accuracy (mean)": r["cv_score"],
            "CV Accuracy (std)": 0,
            "Test Accuracy": r["test_acc"],
            "Test F1 (weighted)": r["test_f1"],
            "Train Time (s)": 0,
        }
    )

final_results.append(
    {
        "Model": f"{best_name} (calibrated)",
        "CV Accuracy (mean)": 0,
        "CV Accuracy (std)": 0,
        "Test Accuracy": round(cal_acc, 4),
        "Test F1 (weighted)": round(cal_f1, 4),
        "Train Time (s)": 0,
    }
)

# Majority baseline
final_results.append(
    {
        "Model": "Majority Baseline",
        "CV Accuracy (mean)": round(max(np.bincount(y_train)) / len(y_train), 4),
        "CV Accuracy (std)": 0,
        "Test Accuracy": round(max(np.bincount(y_test)) / len(y_test), 4),
        "Test F1 (weighted)": 0,
        "Train Time (s)": 0,
    }
)

# Classification report
y_pred_best = best_pipeline.predict(X_test_final)
class_names = [str(c) for c in target_encoder.classes_]
print("=" * 70)
print("CLASSIFICATION REPORT (Best Model)")
print("=" * 70)
print(classification_report(y_test, y_pred_best, target_names=class_names))

print("\n(Base comparison table built — AutoML row will be appended after Section 11.)")

CLASSIFICATION REPORT (Best Model)
                    precision    recall  f1-score   support

 Likely To Connect       0.98      0.97      0.97      2000
   Mostly Browsing       0.95      0.95      0.95      1952
Needs Profile Help       0.97      0.96      0.97      2000
     Ready To Chat       0.96      0.97      0.97      2084
 Swipes Too Freely       0.96      0.97      0.96      1964

          accuracy                           0.97     10000
         macro avg       0.97      0.97      0.97     10000
      weighted avg       0.97      0.97      0.97     10000


(Base comparison table built — AutoML row will be appended after Section 11.)


## 12. AutoML Baseline Comparison

We run [auto-sklearn](https://automl.github.io/auto-sklearn/) on the same train/test split to provide a direct answer to the rubric question: *"How does your model compare to auto-sklearn?"*

auto-sklearn automates model selection and hyperparameter tuning via Bayesian optimization over a large portfolio of learners. We pre-resample the training data with SMOTE (since auto-sklearn does not support imblearn pipelines natively) and give it a fixed time budget from `CONFIG["automl"]`. If auto-sklearn cannot be installed (it requires Linux and specific Python/scikit-learn versions), we fall back to [FLAML](https://github.com/microsoft/FLAML) — a lightweight, cross-platform AutoML library by Microsoft.

In [13]:
import subprocess
import sys

from sklearn.metrics import accuracy_score, f1_score
from sklearn.dummy import DummyClassifier
from imblearn.over_sampling import SMOTE

AUTOML_CFG = CONFIG["automl"]

# -------------------------------------------------------------------
# 1. Install AutoML backend
# -------------------------------------------------------------------
automl_backend = None

# auto-sklearn requires Python 3.10 + scikit-learn 0.24.x.
# All current Colab runtimes (2025.07–2026.04) use Python 3.11+,
# so auto-sklearn cannot be installed. We attempt it on Linux and
# fall back to FLAML — a cross-platform AutoML library by Microsoft
# that works with modern Python/scikit-learn.
if sys.platform == "linux":
    try:
        print("Attempting auto-sklearn (requires Python 3.10 + scikit-learn 0.24.x)...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "auto-sklearn"],
            timeout=300,
        )
        import autosklearn.classification
        automl_backend = "auto-sklearn"
        print(f"Installed: {automl_backend}")
    except Exception as e:
        print(f"auto-sklearn unavailable: {e}")

if automl_backend is None:
    print("Using FLAML (Microsoft AutoML — cross-platform, modern Python compatible)...")
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "flaml[automl]"],
            timeout=120,
        )
        from flaml import AutoML
        automl_backend = "FLAML"
        print(f"Installed: {automl_backend}")
    except Exception as e:
        print(f"FLAML also unavailable: {e}")

# -------------------------------------------------------------------
# 2. Pre-resample with SMOTE
# -------------------------------------------------------------------
smote = SMOTE(random_state=SEED)
X_train_res, y_train_res = smote.fit_resample(X_train_final, y_train)
print(f"\nSMOTE resampled: {X_train_res.shape[0]} train samples")

# -------------------------------------------------------------------
# 3. Run AutoML
# -------------------------------------------------------------------
automl_acc = None
automl_f1 = None
automl_time = 0.0

if automl_backend == "auto-sklearn":
    t0 = pd.Timestamp.now()
    automl = autosklearn.classification.AutoSklearnClassifier(
        time_left_for_this_task=AUTOML_CFG["time_budget"],
        per_run_time_limit=max(30, AUTOML_CFG["time_budget"] // 10),
        n_jobs=-1,
        seed=SEED,
        metric=autosklearn.metrics.accuracy,
    )
    automl.fit(X_train_res, y_train_res)
    automl_time = (pd.Timestamp.now() - t0).total_seconds()

    y_automl = automl.predict(X_test_final)
    automl_acc = accuracy_score(y_test, y_automl)
    automl_f1 = f1_score(y_test, y_automl, average="weighted")
    print(f"\nauto-sklearn — Acc: {automl_acc:.4f}, F1: {automl_f1:.4f}, Time: {automl_time:.0f}s")
    print(f"  Models found: {automl.leaderboard().shape[0]}")
    print(automl.leaderboard().head(5).to_string())

elif automl_backend == "FLAML":
    t0 = pd.Timestamp.now()
    automl = AutoML()
    automl.fit(
        X_train_res, y_train_res,
        task="classification",
        time_budget=AUTOML_CFG["time_budget"],
        metric="accuracy",
        seed=SEED,
        verbose=0,
    )
    automl_time = (pd.Timestamp.now() - t0).total_seconds()

    y_automl = automl.predict(X_test_final)
    automl_acc = accuracy_score(y_test, y_automl)
    automl_f1 = f1_score(y_test, y_automl, average="weighted")
    print(f"\nFLAML — Acc: {automl_acc:.4f}, F1: {automl_f1:.4f}, Time: {automl_time:.0f}s")
    print(f"  Best estimator: {automl.best_estimator}")
else:
    print("\nNo AutoML backend available — skipping AutoML comparison.")

# -------------------------------------------------------------------
# 4. Dummy baselines
# -------------------------------------------------------------------
dummy = DummyClassifier(strategy="most_frequent", random_state=SEED)
dummy.fit(X_train_final, y_train)
y_dummy = dummy.predict(X_test_final)
dummy_acc = accuracy_score(y_test, y_dummy)
dummy_f1 = f1_score(y_test, y_dummy, average="weighted")

dummy_uniform = DummyClassifier(strategy="uniform", random_state=SEED)
dummy_uniform.fit(X_train_final, y_train)
y_uniform = dummy_uniform.predict(X_test_final)
uniform_acc = accuracy_score(y_test, y_uniform)
uniform_f1 = f1_score(y_test, y_uniform, average="weighted")

# Untuned best model
untuned_pipe = models[best_name]
untuned_pipe.fit(X_train_final, y_train)
y_untuned = untuned_pipe.predict(X_test_final)
untuned_acc = accuracy_score(y_test, y_untuned)
untuned_f1 = f1_score(y_test, y_untuned, average="weighted")

# -------------------------------------------------------------------
# 5. Side-by-side comparison
# -------------------------------------------------------------------
print("\n" + "=" * 70)
print("MODEL EVALUATION: Manual Pipeline vs. AutoML vs. Baselines")
print("=" * 70)

comparison_rows = [
    ("Majority Baseline", dummy_acc, dummy_f1, 0),
    ("Random Baseline", uniform_acc, uniform_f1, 0),
    (f"Untuned {best_name}", untuned_acc, untuned_f1, 0),
    (f"{best_name} (tuned + calibrated)", cal_acc, cal_f1, 0),
]
if automl_acc is not None:
    comparison_rows.append(
        (f"{automl_backend} (AutoML)", automl_acc, automl_f1, automl_time)
    )

comparison_df = pd.DataFrame(
    comparison_rows,
    columns=["Model", "Test Accuracy", "Test F1 (weighted)", "Time (s)"],
).sort_values("Test Accuracy", ascending=False).reset_index(drop=True)

print(comparison_df.to_string(index=False))

# Store for the final comparison table
automl_result = {
    "Model": f"{automl_backend} (AutoML)" if automl_backend else None,
    "CV Accuracy (mean)": 0,
    "CV Accuracy (std)": 0,
    "Test Accuracy": round(automl_acc, 4) if automl_acc else None,
    "Test F1 (weighted)": round(automl_f1, 4) if automl_f1 else None,
    "Train Time (s)": round(automl_time, 1),
}

print("\n" + "=" * 70)
print("EVALUATION CONCLUSION")
print("=" * 70)
if automl_acc is not None:
    diff = cal_acc - automl_acc
    direction = "outperforms" if diff > 0 else "underperforms" if diff < 0 else "matches"
    print(
        f"Our tuned {best_name} {direction} {automl_backend} by {abs(diff):.4f} accuracy."
    )
    print(
        f"  Manual pipeline:  Acc={cal_acc:.4f}, F1={cal_f1:.4f}"
    )
    print(
        f"  {automl_backend:15s}  Acc={automl_acc:.4f}, F1={automl_f1:.4f}"
    )
    print()
    print(
        "Key advantage: our manual pipeline provides SHAP explanations, "
        "calibration plots, domain-specific feature engineering, and full "
        "control over every stage — capabilities AutoML does not offer."
    )
else:
    print("AutoML comparison skipped (no backend available).")

Using FLAML (Microsoft AutoML — cross-platform, modern Python compatible)...
Installed: FLAML

SMOTE resampled: 41675 train samples


c:\WID3006_ML\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(



FLAML — Acc: 0.9669, F1: 0.9669, Time: 343s
  Best estimator: catboost

MODEL EVALUATION: Manual Pipeline vs. AutoML vs. Baselines
                        Model  Test Accuracy  Test F1 (weighted)   Time (s)
               FLAML (AutoML)         0.9669            0.966914 342.887714
CatBoost (tuned + calibrated)         0.9644            0.964408   0.000000
             Untuned CatBoost         0.9580            0.958009   0.000000
            Majority Baseline         0.2084            0.071881   0.000000
              Random Baseline         0.1953            0.195304   0.000000

EVALUATION CONCLUSION
Our tuned CatBoost underperforms FLAML by 0.0025 accuracy.
  Manual pipeline:  Acc=0.9644, F1=0.9644
  FLAML            Acc=0.9669, F1=0.9669

Key advantage: our manual pipeline provides SHAP explanations, calibration plots, domain-specific feature engineering, and full control over every stage — capabilities AutoML does not offer.


## 13. Final Comparison Table

All models — base, tuned, calibrated, majority baseline, and AutoML — are compared side by side.

In [14]:
# -------------------------------------------------------------------
# Final comparison table — includes all models + AutoML
# -------------------------------------------------------------------
if automl_result["Model"] is not None:
    final_results.append(automl_result)

final_df = (
    pd.DataFrame(final_results)
    .sort_values("Test Accuracy", ascending=False)
    .reset_index(drop=True)
)

print("=" * 70)
print("FINAL MODEL COMPARISON")
print("=" * 70)
print(final_df.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(
    x="Test Accuracy",
    y="Model",
    data=final_df,
    palette="viridis",
    ax=axes[0],
    hue="Model",
    legend=False,
)
axes[0].set_title("Test Accuracy by Model", fontsize=14)
axes[0].set_xlim(final_df["Test Accuracy"].min() - 0.02, 1.0)

nonzero_f1 = final_df[final_df["Test F1 (weighted)"] > 0].copy()
if len(nonzero_f1) > 0:
    sns.barplot(
        x="Test F1 (weighted)",
        y="Model",
        data=nonzero_f1,
        palette="magma",
        ax=axes[1],
        hue="Model",
        legend=False,
    )
    axes[1].set_title("Test F1 (Weighted) by Model", fontsize=14)

plt.tight_layout()
plt.show()

print("\nPipeline complete! Best model:", best_name)

FINAL MODEL COMPARISON
                Model  CV Accuracy (mean)  CV Accuracy (std)  Test Accuracy  Test F1 (weighted)  Train Time (s)
       FLAML (AutoML)              0.0000             0.0000         0.9669              0.9669           342.9
     CatBoost (tuned)              0.9585             0.0000         0.9653              0.9653             0.0
CatBoost (calibrated)              0.0000             0.0000         0.9644              0.9644             0.0
             CatBoost              0.9529             0.0013         0.9580              0.9580            46.6
             LightGBM              0.9510             0.0033         0.9553              0.9553            36.1
     LightGBM (tuned)              0.9494             0.0000         0.9552              0.9552             0.0
              XGBoost              0.9514             0.0029         0.9552              0.9552            39.3
      XGBoost (tuned)              0.9511             0.0000         0.9547      

C:\Users\tzx20\AppData\Local\Temp\ipykernel_31068\130213379.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 14. Save Artifacts

In [15]:
RESULTS_DIR = Path("ML_Results")
RESULTS_DIR.mkdir(exist_ok=True)

joblib.dump(calibrated_pipeline, RESULTS_DIR / "best_tuned_model.pkl")
joblib.dump(target_encoder, RESULTS_DIR / "target_encoder.pkl")
joblib.dump(selected_features, RESULTS_DIR / "selected_features.pkl")
joblib.dump(scaler, RESULTS_DIR / "scaler.pkl")
final_df.to_csv(RESULTS_DIR / "final_comparison.csv", index=False)

# Save classification report
with open(RESULTS_DIR / "classification_report.txt", "w") as f:
    f.write(f"Best Model: {best_name}\n\n")
    f.write(classification_report(y_test, y_pred_best, target_names=class_names))

print("Artifacts saved to ML_Results/")
for f in sorted(RESULTS_DIR.iterdir()):
    print(f"  {f.name}")

Artifacts saved to ML_Results/
  best_tuned_model.pkl
  classification_report.txt
  final_comparison.csv
  scaler.pkl
  selected_features.pkl
  target_encoder.pkl


## 15. Download Artifacts (Optional)

Download the trained model and artifacts to use locally or deploy.

In [16]:
# Uncomment to download artifacts
# import shutil
# shutil.make_archive("ML_Results", "zip", "ML_Results")
# shutil.make_archive("Preprocessed_Data_V2", "zip", "Preprocessed_Data_V2")
# from google.colab import files
# files.download("ML_Results.zip")
# files.download("Preprocessed_Data_V2.zip")

## 16. Streamlit Dashboard (Optional)

Generates and launches a self-contained ConnectionLens dashboard — 
no external files required. The dashboard includes:
- **Scenario Predictor** — adjust profile sliders and see the predicted connection stage
- **Model Comparison** — bar charts of all trained models
- **Feature Importance** — top features from the trained model

**On Google Colab:** A public URL will be generated via localtunnel.
**On local Jupyter:** Open the printed localhost URL in your browser.

In [ ]:
# ---------------------------------------------------------------------------
# Launch a self-contained ConnectionLens Streamlit dashboard
# ---------------------------------------------------------------------------
import subprocess, sys, time, textwrap, os
from pathlib import Path

# Install streamlit + plotly if needed
try:
    import streamlit
    print(f"streamlit {streamlit.__version__} already installed.")
except ImportError:
    print("Installing streamlit + plotly...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "streamlit", "plotly"])

# --- Generate self-contained dashboard script ---
DASHBOARD_CODE = textwrap.dedent("""\
    \"\"\"ConnectionLens Dashboard — self-contained (generated by notebook).\"\"\"
    from pathlib import Path
    import joblib, numpy as np, pandas as pd
    import plotly.express as px, plotly.graph_objects as go
    import streamlit as st

    # -- Inline connection scoring helpers --
    def _numeric(frame, column, default=0.0):
        if column not in frame.columns:
            return pd.Series(default, index=frame.index, dtype="float64")
        return pd.to_numeric(frame[column], errors="coerce").fillna(default)

    def _bounded(series, cap):
        values = pd.to_numeric(series, errors="coerce").fillna(0).clip(lower=0)
        return (values / cap).clip(0, 1)

    def _num_interests(frame):
        if "interest_tags" in frame.columns:
            tags = frame["interest_tags"].fillna("").astype(str).str.split(r",\\s*")
            return tags.apply(lambda values: len([tag for tag in values if tag]))
        return pd.Series(0, index=frame.index, dtype="float64")

    def add_connection_features(frame, config):
        cfg = config
        engineered = frame.copy()
        app_usage = _numeric(engineered, "app_usage_time_min")
        swipe_ratio = _numeric(engineered, "swipe_right_ratio")
        likes = _numeric(engineered, "likes_received")
        matches = _numeric(engineered, "mutual_matches")
        messages = _numeric(engineered, "message_sent_count")
        emoji_rate = _numeric(engineered, "emoji_usage_rate")
        bio_length = _numeric(engineered, "bio_length")
        profile_pics = _numeric(engineered, "profile_pics_count")
        height = _numeric(engineered, "height_cm")
        weight = _numeric(engineered, "weight_kg")
        pc_caps = cfg["profile_completeness"]["caps"]
        pc_wts = cfg["profile_completeness"]["weights"]
        engineered["num_interests"] = _num_interests(engineered)
        engineered["match_rate"] = matches / (likes + 1)
        engineered["msg_per_match"] = messages / (matches + 1)
        if "height_cm" in frame.columns and "weight_kg" in frame.columns:
            engineered["bmi"] = np.where(height > 0, weight / ((height / 100) ** 2), 0)
        engineered["profile_completeness"] = (
            profile_pics.clip(0, pc_caps["profile_pics"]) / pc_caps["profile_pics"] * pc_wts["profile_pics"]
            + bio_length.clip(0, pc_caps["bio_length"]) / pc_caps["bio_length"] * pc_wts["bio_length"]
            + engineered["num_interests"].clip(0, pc_caps["interests"]) / pc_caps["interests"] * pc_wts["interests"]
        )
        ideal_ratio = cfg["selectivity_balance"]["ideal_swipe_ratio"]
        swipe_thresh = cfg["swipe_excess"]["threshold"]
        engineered["selectivity_balance"] = (1 - (swipe_ratio - ideal_ratio).abs() / ideal_ratio).clip(0, 1)
        engineered["swipe_excess"] = (swipe_ratio - swipe_thresh).clip(lower=0)
        engineered["like_to_match_gap"] = (likes - matches).clip(lower=0)
        engineered["conversation_depth"] = np.log1p(messages) * np.log1p(engineered["msg_per_match"])
        engineered["social_pull"] = likes / (profile_pics + 1)
        engineered["activity_level"] = np.log1p(app_usage)
        if "last_active_hour" in engineered.columns:
            hour = _numeric(engineered, "last_active_hour") % 24
            engineered["last_active_sin"] = np.sin(2 * np.pi * hour / 24)
            engineered["last_active_cos"] = np.cos(2 * np.pi * hour / 24)
        mq_caps = cfg["match_quality"]["caps"]
        mq_wts = cfg["match_quality"]["weights"]
        engineered["match_quality"] = (
            mq_wts["match_rate"] * engineered["match_rate"].clip(0, 1)
            + mq_wts["matches"] * _bounded(matches, mq_caps["matches"])
            + mq_wts["selectivity"] * engineered["selectivity_balance"]
            + mq_wts["social_pull"] * _bounded(engineered["social_pull"], mq_caps["social_pull"])
        )
        cq_caps = cfg["conversation_quality"]["caps"]
        cq_wts = cfg["conversation_quality"]["weights"]
        engineered["conversation_quality"] = (
            cq_wts["msg_per_match"] * _bounded(engineered["msg_per_match"], cq_caps["msg_per_match"])
            + cq_wts["messages"] * _bounded(messages, cq_caps["messages"])
            + cq_wts["emoji_rate"] * _bounded(emoji_rate, cq_caps["emoji_rate"])
            + cq_wts["app_usage"] * _bounded(app_usage, cq_caps["app_usage"])
        )
        pq_caps = cfg["profile_quality"]["caps"]
        pq_wts = cfg["profile_quality"]["weights"]
        engineered["profile_quality"] = (
            pq_wts["completeness"] * engineered["profile_completeness"]
            + pq_wts["bio_length"] * _bounded(bio_length, pq_caps["bio_length"])
            + pq_wts["profile_pics"] * _bounded(profile_pics, pq_caps["profile_pics"])
        )
        cs_caps = cfg["connection_score"]["caps"]
        cs_wts = cfg["connection_score"]["weights"]
        engineered["connection_score"] = (
            cs_wts["match_quality"] * engineered["match_quality"]
            + cs_wts["conversation_quality"] * engineered["conversation_quality"]
            + cs_wts["profile_quality"] * engineered["profile_quality"]
            + cs_wts["activity_level"] * _bounded(app_usage, cs_caps["app_usage"])
            - cs_wts["swipe_penalty"] * _bounded(engineered["swipe_excess"], cs_caps["swipe_excess"])
        )
        bi_caps = cfg["browser_issue"]["caps"]
        bi_wts = cfg["browser_issue"]["weights"]
        engineered["browser_issue"] = (
            bi_wts["low_app_usage"] * (1 - _bounded(app_usage, bi_caps["app_usage"]))
            + bi_wts["low_messages"] * (1 - _bounded(messages, bi_caps["messages"]))
            + bi_wts["low_matches"] * (1 - _bounded(matches, bi_caps["matches"]))
        )
        si_caps = cfg["swipe_issue"]["caps"]
        si_wts = cfg["swipe_issue"]["weights"]
        engineered["swipe_issue"] = (
            si_wts["swipe_excess"] * _bounded(engineered["swipe_excess"], si_caps["swipe_excess"])
            + si_wts["low_match_rate"] * (1 - engineered["match_rate"].clip(0, 1))
        )
        return engineered

    # -- Scoring config (mirrors notebook CONFIG["scoring"]) --
    SCORING = {
        "profile_completeness": {"caps": {"profile_pics": 6, "bio_length": 300, "interests": 5}, "weights": {"profile_pics": 0.40, "bio_length": 0.40, "interests": 0.20}},
        "selectivity_balance": {"ideal_swipe_ratio": 0.55},
        "swipe_excess": {"threshold": 0.70},
        "match_quality": {"caps": {"matches": 40, "social_pull": 50}, "weights": {"match_rate": 0.45, "matches": 0.25, "selectivity": 0.15, "social_pull": 0.15}},
        "conversation_quality": {"caps": {"msg_per_match": 10, "messages": 80, "emoji_rate": 1, "app_usage": 300}, "weights": {"msg_per_match": 0.40, "messages": 0.30, "emoji_rate": 0.20, "app_usage": 0.10}},
        "profile_quality": {"caps": {"bio_length": 300, "profile_pics": 6}, "weights": {"completeness": 0.60, "bio_length": 0.25, "profile_pics": 0.15}},
        "connection_score": {"weights": {"match_quality": 0.35, "conversation_quality": 0.30, "profile_quality": 0.20, "activity_level": 0.15, "swipe_penalty": 0.10}, "caps": {"app_usage": 300, "swipe_excess": 0.30}},
        "browser_issue": {"caps": {"app_usage": 300, "messages": 80, "matches": 40}, "weights": {"low_app_usage": 0.45, "low_messages": 0.35, "low_matches": 0.20}},
        "swipe_issue": {"caps": {"swipe_excess": 0.30}, "weights": {"swipe_excess": 0.55, "low_match_rate": 0.45}},
        "stage_thresholds": {"likely_to_connect": 0.80, "needs_profile_help": 0.20, "mostly_browsing_quantile": 0.62, "swipes_too_freely_quantile": 0.50},
    }

    STAGE_COLORS = {
        "Likely To Connect": "#2ecc71", "Ready To Chat": "#27ae60",
        "Mostly Browsing": "#f39c12", "Swipes Too Freely": "#e74c3c",
        "Needs Profile Help": "#c0392b",
    }

    # -- Paths --
    ROOT = Path.cwd()
    PREP = ROOT / "Preprocessed_Data_V2"
    RES = ROOT / "ML_Results"

    # -- Page config --
    st.set_page_config(page_title="ConnectionLens", page_icon=":bar_chart:", layout="wide")

    @st.cache_resource
    def load_artifacts():
        a = {}
        for name in ["target_encoder", "scaler", "selected_features"]:
            p = PREP / f"{name}.pkl"
            if p.exists(): a[name] = joblib.load(p)
        p = RES / "best_tuned_model.pkl"
        if p.exists(): a["model"] = joblib.load(p)
        return a

    @st.cache_data
    def load_comparison():
        p = RES / "final_comparison.csv"
        return pd.read_csv(p) if p.exists() else None

    # -- Sidebar navigation --
    page = st.sidebar.radio("Navigation", ["Scenario Predictor", "Model Comparison", "Feature Importance"])

    artifacts = load_artifacts()

    # ===================================================================
    # PAGE: Scenario Predictor
    # ===================================================================
    if page == "Scenario Predictor":
        st.title("Connection Readiness Predictor")
        if "model" not in artifacts:
            st.error("Model not found. Run the ML pipeline first.")
            st.stop()
        model = artifacts["model"]
        encoder = artifacts["target_encoder"]
        scaler = artifacts["scaler"]
        selected_features = artifacts["selected_features"]
        full_columns = list(scaler.feature_names_in_) if hasattr(scaler, "feature_names_in_") else list(selected_features)

        col_l, col_r = st.columns(2)
        with col_l:
            st.subheader("Profile Inputs")
            app_usage = st.slider("App Usage (min/day)", 0, 1000, 120)
            swipe_ratio = st.slider("Swipe Right Ratio", 0.0, 1.0, 0.5, 0.01)
            likes_received = st.slider("Likes Received", 0, 10000, 50)
            mutual_matches = st.slider("Mutual Matches", 0, 10000, 10)
            msg_sent = st.slider("Messages Sent", 0, 10000, 30)
            bio_length = st.slider("Bio Length (chars)", 0, 5000, 140)
        with col_r:
            st.subheader("Physical & Activity")
            emoji_rate = st.slider("Emoji Usage Rate", 0.0, 10.0, 0.3, 0.1)
            height_cm = st.slider("Height (cm)", 80, 250, 170)
            weight_kg = st.slider("Weight (kg)", 20, 300, 70)
            profile_pics = st.slider("Profile Pictures", 0, 50, 3)
            last_active = st.slider("Last Active Hour", 0, 23, 12)

        input_df = pd.DataFrame(np.zeros((1, len(full_columns))), columns=full_columns)
        values = {"app_usage_time_min": app_usage, "swipe_right_ratio": swipe_ratio, "likes_received": likes_received, "mutual_matches": mutual_matches, "message_sent_count": msg_sent, "bio_length": bio_length, "emoji_usage_rate": emoji_rate, "height_cm": height_cm, "weight_kg": weight_kg, "profile_pics_count": profile_pics, "last_active_hour": last_active}
        for col, val in values.items():
            if col in input_df.columns: input_df.at[0, col] = val
        engineered = add_connection_features(input_df, SCORING)
        for col in full_columns:
            if col in engineered.columns: input_df[col] = engineered[col]
        scaled = pd.DataFrame(scaler.transform(input_df), columns=full_columns)
        selected = scaled[selected_features]
        encoded = model.predict(selected).astype(int)
        label = encoder.inverse_transform(encoded)[0]

        st.markdown("---")
        colors = {"Likely To Connect": "success", "Ready To Chat": "success", "Mostly Browsing": "warning", "Swipes Too Freely": "error", "Needs Profile Help": "info"}
        getattr(st, colors.get(label, "info"))(f"**Predicted Stage: {label}**")

        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(selected)
            prob_df = pd.DataFrame({"Class": encoder.inverse_transform(range(proba.shape[1])), "Probability": proba[0]}).sort_values("Probability", ascending=False)
            col_a, col_b = st.columns(2)
            with col_a:
                st.subheader("Class Probabilities")
                fig = px.bar(prob_df, x="Probability", y="Class", orientation="h", color="Class", color_discrete_map=STAGE_COLORS)
                fig.update_layout(height=300, xaxis_range=[0, 1], showlegend=False)
                st.plotly_chart(fig, use_container_width=True)
            with col_b:
                confidence = float(np.max(proba))
                gauge_color = STAGE_COLORS.get(label, "#95a5a6")
                fig = go.Figure(go.Indicator(mode="gauge+number", value=confidence * 100, title={"text": "Confidence (%)"}, gauge={"axis": {"range": [0, 100]}, "bar": {"color": gauge_color}, "steps": [{"range": [0, 50], "color": "lightcoral"}, {"range": [50, 80], "color": "lightyellow"}, {"range": [80, 100], "color": "lightgreen"}]}))
                fig.update_layout(height=300)
                st.plotly_chart(fig, use_container_width=True)

        # Sensitivity analysis
        st.subheader("Sensitivity Analysis")
        st.markdown("What single change would most affect the prediction?")
        base_proba = model.predict_proba(selected)[0] if hasattr(model, "predict_proba") else None
        if base_proba is not None:
            sens_records = []
            for feat_name, feat_val in values.items():
                pct = 0.10
                up = feat_val * (1 + pct) if feat_val != 0 else 1.0
                down = feat_val * (1 - pct) if feat_val != 0 else -1.0
                for direction, new_val in [("up", up), ("down", down)]:
                    test = input_df.copy()
                    if feat_name in test.columns: test.at[0, feat_name] = new_val
                    test_eng = add_connection_features(test, SCORING)
                    for c in full_columns:
                        if c in test_eng.columns: test[c] = test_eng[c]
                    test_scaled = pd.DataFrame(scaler.transform(test), columns=full_columns)
                    test_sel = test_scaled[selected_features]
                    test_proba = model.predict_proba(test_sel)[0]
                    sens_records.append({"Feature": feat_name, "Direction": direction, "Probability Change": float(np.max(test_proba) - np.max(base_proba))})
            sens_df = pd.DataFrame(sens_records)
            agg = sens_df.groupby("Feature")["Probability Change"].apply(lambda x: x.abs().max()).reset_index()
            agg.columns = ["Feature", "Max Impact"]
            agg = agg.sort_values("Max Impact", ascending=True).tail(10)
            fig = px.bar(agg, x="Max Impact", y="Feature", orientation="h", color="Max Impact", color_continuous_scale="OrRd")
            fig.update_layout(height=350, xaxis_title="Max probability change (+-10% perturbation)")
            st.plotly_chart(fig, use_container_width=True)

    # ===================================================================
    # PAGE: Model Comparison
    # ===================================================================
    elif page == "Model Comparison":
        st.title("Model Comparison")
        comp = load_comparison()
        if comp is None:
            st.error("No comparison data found. Run the ML pipeline first.")
            st.stop()
        st.dataframe(comp, use_container_width=True)
        col1, col2 = st.columns(2)
        with col1:
            fig = px.bar(comp, x="Test Accuracy", y="Model", orientation="h", color="Test Accuracy", color_continuous_scale="Viridis")
            fig.update_layout(height=400, showlegend=False)
            st.plotly_chart(fig, use_container_width=True)
        with col2:
            nonzero = comp[comp["Test F1 (weighted)"] > 0]
            if len(nonzero) > 0:
                fig = px.bar(nonzero, x="Test F1 (weighted)", y="Model", orientation="h", color="Test F1 (weighted)", color_continuous_scale="Magma")
                fig.update_layout(height=400, showlegend=False)
                st.plotly_chart(fig, use_container_width=True)

    # ===================================================================
    # PAGE: Feature Importance
    # ===================================================================
    elif page == "Feature Importance":
        st.title("Feature Importance")
        if "model" not in artifacts:
            st.error("Model not found.")
            st.stop()
        model = artifacts["model"]
        base = model.estimator if hasattr(model, "estimator") else model
        inner = base.named_steps["model"] if hasattr(base, "named_steps") else base
        if hasattr(inner, "feature_importances_"):
            feats = artifacts["selected_features"]
            imp_df = pd.DataFrame({"Feature": feats, "Importance": inner.feature_importances_}).sort_values("Importance", ascending=False).head(20)
            fig = px.bar(imp_df, x="Importance", y="Feature", orientation="h", color="Importance", color_continuous_scale="Viridis")
            fig.update_layout(height=500)
            st.plotly_chart(fig, use_container_width=True)
        else:
            st.info("Feature importances not available for this model type.")
""")

# Write dashboard to temp file
_dashboard_path = Path("_connectionlens_dashboard.py")
_dashboard_path.write_text(DASHBOARD_CODE, encoding="utf-8")
print(f"Dashboard script written to {_dashboard_path}")

# Launch
PORT = 8501
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    import threading
    def _run():
        subprocess.run([sys.executable, "-m", "streamlit", "run", str(_dashboard_path),
            "--server.port", str(PORT), "--server.headless", "true",
            "--browser.gatherUsageStats", "false"], check=False)
    threading.Thread(target=_run, daemon=True).start()
    time.sleep(5)
    subprocess.check_call(["npm", "install", "-g", "localtunnel"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"\nPublic URL: https://loca.lt:{PORT}")
    print("If the URL does not work, wait a few seconds and refresh.\n")
    subprocess.run(["lt", "--port", str(PORT)], check=False)
else:
    print(f"\nOpen in browser: http://localhost:{PORT}\n")
    print("To stop: Interrupt kernel (Ctrl+C).\n")
    subprocess.run([sys.executable, "-m", "streamlit", "run", str(_dashboard_path),
        "--server.port", str(PORT), "--server.headless", "true",
        "--browser.gatherUsageStats", "false"], check=False)

streamlit 1.57.0 already installed.
Dashboard script written to _connectionlens_dashboard.py

Open in browser: http://localhost:8501

To stop: Interrupt kernel (Ctrl+C).

